In [ ]:
import glob, pathlib, json

In [ ]:
fns = glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\cad\*\*")
has_tree = []
empty = []
for fn in fns:
    p = pathlib.Path(fn) / "artifacts" / "object_list.json"
    if not p.exists():
        continue
    with open(p, "r") as f:
        d = json.load(f)
        if len(d["needed_objects"]) == 0:
            empty.append(fn)
        if "meta_links" not in d:
            continue
        if any("collision" in v for v in d["meta_links"].values()):
            has_tree.append(fn)

In [ ]:
print(
    "dvc unprotect",
    " ".join(
        str(pathlib.Path(x).relative_to(pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline")))
        for x in empty
    ),
)

In [ ]:
print(
    "\n".join(
        "rm -Recurse "
        + str(
            pathlib.Path(x).relative_to(pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline"))
        )
        for x in empty
    )
)

In [ ]:
len(has_tree)

In [ ]:
has_tree[0]

In [ ]:
missing = set(fns) - set(has_tree)

In [ ]:
len(missing)

In [ ]:
target_fns = []
for m in sorted(missing)[:100]:
    fn = pathlib.Path(m) / "artifacts/object_list.json"
    if not fn.exists():
        continue
    relative_fn = fn.relative_to(pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline"))
    target_fns.append(str(relative_fn))

print("dvc unprotect", " ".join(target_fns))

In [ ]:
for m in sorted(missing)[:100]:
    fn = pathlib.Path(m) / "processed.max"
    relative_fn = fn.relative_to(pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline"))
    print(
        f"python b1k_pipeline/batch_3dsmax.py {relative_fn} b1k_pipeline/max/object_list.py"
    )

In [ ]:
to_commit = []
for fn in fns:
    p = pathlib.Path(fn) / "artifacts" / "object_list.json"
    if p.exists() and not p.is_symlink():
        relative_fn = p.relative_to(pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline"))
        to_commit.append(str(relative_fn).replace("\\", "/"))
print("dvc commit", " ".join(to_commit[:50]))

In [ ]:
# Get all the object targets, split into two, and run a bunch of them
import sys

sys.path.append(r"D:\BEHAVIOR-1K\asset_pipeline")
from b1k_pipeline.utils import get_targets

t = get_targets("scenes")

import random

random.seed(1234)
random.shuffle(t)
assert "objects/compost-pf" not in t

In [ ]:
i = 1
side = t[i::2]
for start in range(0, len(side), 100):
    batch = side[start : start + 100]
    print("dvc repro", " ".join(f"export_meshes@{x}" for x in batch))
print("")

In [ ]:
len(glob.glob(r"D:\BEHAVIOR-1K\asset_pipeline\cad\scenes\*\artifacts\meshes.zip"))

In [ ]:
[
    x
    for x in t
    if not (
        pathlib.Path(r"D:\BEHAVIOR-1K\asset_pipeline\cad") / x / "artifacts/meshes.zip"
    ).exists()
]